# API Integration Examples

This notebook demonstrates how to interact with the Fairness Project REST API.

## Prerequisites

Start the API server before running this notebook:

```bash
MODEL_PATH=models/model.joblib uvicorn fairness_project.inference.api:app --host 0.0.0.0 --port 8000
```

## Table of Contents

1. [Setup](#setup)
2. [Health Check](#health-check)
3. [Metadata Retrieval](#metadata-retrieval)
4. [Single Prediction](#single-prediction)
5. [Batch Prediction](#batch-prediction)
6. [Pandas Integration](#pandas-integration)
7. [Error Handling](#error-handling)
8. [cURL Examples](#curl-examples)
9. [Troubleshooting](#troubleshooting)

In [ ]:
import json

import pandas as pd
import requests

BASE_URL = "http://localhost:8000"
print(f"API base URL: {BASE_URL}")

## Health Check <a id='health-check'></a>

Verify the server is running and the model is loaded.

In [ ]:
resp = requests.get(f"{BASE_URL}/health")
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Metadata Retrieval <a id='metadata-retrieval'></a>

Retrieve model version, training info, and fairness metrics. Useful for governance threshold checks before making predictions.

In [ ]:
resp = requests.get(f"{BASE_URL}/v1/metadata")
metadata = resp.json()
print(json.dumps(metadata, indent=2))

# Check governance thresholds
fairness = metadata.get("fairness_metrics", {})
if fairness:
    tpr_gap = fairness.get("TPR_gap", None)
    if tpr_gap is not None and abs(tpr_gap) > 0.05:
        print(f"\nWARNING: TPR gap ({tpr_gap:.4f}) exceeds threshold (0.05)")
    else:
        print(f"\nTPR gap within governance threshold")

## Single Prediction <a id='single-prediction'></a>

Submit a single candidate for prediction.

In [ ]:
candidate = {
    "age": 35,
    "workclass": "Private",
    "fnlwgt": 200000,
    "education": "Bachelors",
    "education_num": 13,
    "marital_status": "Married-civ-spouse",
    "occupation": "Exec-managerial",
    "relationship": "Husband",
    "native_country": "United-States",
    "capital_gain": 5000,
    "capital_loss": 0,
    "hours_per_week": 40,
}

resp = requests.post(f"{BASE_URL}/v1/predict", json=candidate)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Batch Prediction <a id='batch-prediction'></a>

Submit multiple candidates in a single request.

In [ ]:
batch = {
    "instances": [
        {
            "age": 35,
            "workclass": "Private",
            "fnlwgt": 200000,
            "education": "Bachelors",
            "education_num": 13,
            "marital_status": "Married-civ-spouse",
            "occupation": "Exec-managerial",
            "relationship": "Husband",
            "native_country": "United-States",
            "capital_gain": 5000,
            "capital_loss": 0,
            "hours_per_week": 40,
        },
        {
            "age": 28,
            "workclass": "State-gov",
            "fnlwgt": 150000,
            "education": "Masters",
            "education_num": 14,
            "marital_status": "Never-married",
            "occupation": "Prof-specialty",
            "relationship": "Not-in-family",
            "native_country": "United-States",
            "capital_gain": 0,
            "capital_loss": 0,
            "hours_per_week": 50,
        },
    ]
}

resp = requests.post(f"{BASE_URL}/v1/predict-batch", json=batch)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Pandas Integration <a id='pandas-integration'></a>

Send a DataFrame to the API and merge predictions back.

In [ ]:
df = pd.DataFrame(
    [
        {
            "age": 35,
            "workclass": "Private",
            "fnlwgt": 200000,
            "education": "Bachelors",
            "education_num": 13,
            "marital_status": "Married-civ-spouse",
            "occupation": "Exec-managerial",
            "relationship": "Husband",
            "native_country": "United-States",
            "capital_gain": 5000,
            "capital_loss": 0,
            "hours_per_week": 40,
        },
        {
            "age": 28,
            "workclass": "State-gov",
            "fnlwgt": 150000,
            "education": "Masters",
            "education_num": 14,
            "marital_status": "Never-married",
            "occupation": "Prof-specialty",
            "relationship": "Not-in-family",
            "native_country": "United-States",
            "capital_gain": 0,
            "capital_loss": 0,
            "hours_per_week": 50,
        },
    ]
)

# Send as batch
payload = {"instances": df.to_dict(orient="records")}
resp = requests.post(f"{BASE_URL}/v1/predict-batch", json=payload)
preds = resp.json()["predictions"]

# Merge predictions into DataFrame
df["prediction"] = [p["prediction"] for p in preds]
df["probability"] = [p["probability"] for p in preds]
df["label"] = [p["label"] for p in preds]

df

## Error Handling <a id='error-handling'></a>

Demonstrate how the API handles invalid input.

In [ ]:
# Invalid input: age exceeds maximum (120)
invalid_candidate = {
    "age": 150,
    "workclass": "Private",
    "fnlwgt": 200000,
    "education": "Bachelors",
    "education_num": 13,
    "marital_status": "Married-civ-spouse",
    "occupation": "Exec-managerial",
    "relationship": "Husband",
    "native_country": "United-States",
    "capital_gain": 5000,
    "capital_loss": 0,
    "hours_per_week": 40,
}

resp = requests.post(f"{BASE_URL}/v1/predict", json=invalid_candidate)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## cURL Examples <a id='curl-examples'></a>

Copy-paste ready examples for command-line usage.

### Health Check

```bash
curl http://localhost:8000/health
```

### Single Prediction

```bash
curl -X POST http://localhost:8000/v1/predict \
  -H "Content-Type: application/json" \
  -d '{
    "age": 35,
    "workclass": "Private",
    "fnlwgt": 200000,
    "education": "Bachelors",
    "education_num": 13,
    "marital_status": "Married-civ-spouse",
    "occupation": "Exec-managerial",
    "relationship": "Husband",
    "native_country": "United-States",
    "capital_gain": 5000,
    "capital_loss": 0,
    "hours_per_week": 40
  }'
```

### Metadata

```bash
curl http://localhost:8000/v1/metadata
```

## Troubleshooting <a id='troubleshooting'></a>

### Connection Refused

If you see `ConnectionError: Connection refused`, the API server is not running. Start it with:

```bash
MODEL_PATH=models/model.joblib uvicorn fairness_project.inference.api:app --host 0.0.0.0 --port 8000
```

### 503 Model Not Loaded

The server is running but no model is loaded. Set the `MODEL_PATH` environment variable:

```bash
MODEL_PATH=path/to/your/model.joblib uvicorn fairness_project.inference.api:app --port 8000
```

### 422 Validation Error

The request body does not match the expected schema. Check field names, types, and constraints in the [API spec](../docs/api_spec.md).